# 06/AgentCore Runtime 배포 (프로덕션): 텍스트 분류(intent)

**요약**: 앞서 로컬에서 검증한 Strands 에이전트를 Amazon Bedrock AgentCore Runtime에 배포합니다. 세션 격리와 서버리스 실행을 기본으로 제공합니다.

**목적**: AgentCore는 프레임워크와 모델에 종속되지 않는 서버리스 호스팅으로, 인프라를 직접 관리하지 않고도 에이전트를 운영할 수 있습니다.

**배경**: 에이전트를 직접 서버로 띄우면 스케일링/세션 격리/관측(observability)을 모두 직접 구축해야 합니다. AgentCore가 이 부담을 대신 맡아 줍니다.

> 실제 실행에는 AWS 자격증명과 비용이 필요합니다. 먼저 `DRY_RUN=1`로 파이프라인을 검증하세요.

> 주의: **빠르게 바뀌는 영역**: 이 영역은 변화가 잦으므로, 배포하기 전에 리전 가용성/GA 여부/CLI/SDK 스키마를 반드시 다시 확인하세요 (`# TODO verify`).
> 검증(2026-07): 현행 권장 배포 = **`@aws/agentcore` npm CLI**(`agentcore create/dev/deploy/invoke`).
> 구 `bedrock-agentcore-starter-toolkit`(agentcore configure/launch)는 더 이상 권장되지 않습니다.
> 호스팅 SDK = `bedrock-agentcore`(`BedrockAgentCoreApp`/`@app.entrypoint`), ARM64 `/invocations`+`/ping` :8080.

이 트랙에서 만든 tool과 endpoint에 맞춰 `agentcore/app.py`의 `SLM_ENDPOINT_NAME`과 tool 정의를 조정하세요. 함께 제공되는 스캐폴드는 정보추출 트랙을 기준으로 작성되어 있습니다.

## 0. 사전 준비: VS Code **새 터미널**에서 진행하세요
AgentCore는 설치/생성/로컬서버/배포가 모두 CLI 작업이고, 대화형 프롬프트/장시간 dev 서버/PATH 연속성 때문에 **노트북 셀이 아니라 터미널에서** 하는 게 맞습니다. (셀의 `!명령`은 매번 새 셸이라 nvm PATH가 안 이어져 `agentcore: command not found`가 납니다.)

**VS Code에서 새 터미널을 열고**(Terminal → New Terminal), 리포 루트(`sagemaker-finetune-serve-e2e/`)에서 아래를 순서대로 실행하세요:
```bash
# 1) Node ≥ 20 + @aws/agentcore 설치 (Node가 이미 20+면 자동 스킵, sudo 불필요)
bash agentcore/setup_agentcore_cli.sh
source $HOME/.nvm/nvm.sh && nvm use 20      # 이 터미널 세션에 Node 20 적용

# 2) 에이전트 프로젝트 생성 (대화형 마법사 대신 flag 한 방)
bash agentcore/create_agent.sh
```
> `@aws/agentcore`는 **Node.js 20 이상** 필요(18 이하면 `EBADENGINE` + 런타임 오류, `/usr/local` 전역설치는 `EACCES` 권한오류). 위 스크립트가 nvm으로 홈에 Node 20을 깔아 두 문제를 모두 피합니다.
> `create_agent.sh`가 끝나면 SLM tool 이식 결과와 다음 두 단계(로컬 검증 → 배포)를 출력합니다(아래 1~3절과 같은 내용입니다).

## 1. 프로젝트 생성 (non-interactive: 마법사 대신 스크립트 한 방)
`agentcore create`는 대화형으로 하나씩 물어 번거롭습니다. 모든 항목을 flag로 주는 스크립트로 한 번에 생성합니다(실측 2026-07, CLI v0.24.2). **터미널에서** 실행하세요(대화형/PATH 갱신).
```bash
bash agentcore/create_agent.sh
```
> 이 스크립트가 스캐폴딩 생성 + **SLM tool 자동 이식**(데모 tool → `extract_structured_json`, `templates/main.py`) + 모델 ID env화(`templates/load.py`) + `uv sync`까지 한 번에 합니다. 즉 손수 코드 편집 없이 바로 로컬 검증으로 넘어갑니다.

## 2. 로컬 검증 (배포 전 실제 추론: 스크립트 한 방)
배포 전에 **로컬에서 먼저** 에이전트가 도는지 확인합니다(AWS 과금 없음). reasoning은 Bedrock Claude/추출은 SLM endpoint tool로 처리합니다. 아래 스크립트가 dev 서버 기동 → 추론 → 종료를 자동으로 합니다:
```bash
bash agentcore/verify_local.sh <SLM_ENDPOINT_NAME> [AWS_REGION]
# 예: bash agentcore/verify_local.sh gemma-classification-vllm-1784XXXXXX us-west-2
```
> `<SLM_ENDPOINT_NAME>`은 03에서 배포한 endpoint 이름입니다(`%store -r endpoint_name`으로 확인).
> tool은 endpoint에 **`messages` 형식**으로 보냅니다: 그래야 핸들러(`inference.py`)가 서버측에서 chat template을 적용합니다. raw 텍스트를 직송하면 template 미적용으로 빈/degenerate 응답이 납니다(실측).
> 스크립트 내부 실측 노하우: dev 서버는 `setsid ... </dev/null &`로 띄우고(stdin 분리), 종료는 `kill <pid>`로 (`pkill -f 'agentcore dev'`는 실행 셸까지 죽임). 직접 돌릴 일이 있으면 참고하세요.
```bash
# (참고) verify_local.sh가 내부적으로 하는 일:
# export SLM_ENDPOINT_NAME=<endpoint> AWS_REGION=<리전> BEDROCK_CLAUDE_MODEL_ID=global.anthropic.claude-sonnet-5
# cd agentcore/gemmaextraction
# setsid agentcore dev --skip-deploy --logs </dev/null >/tmp/agentcore_dev.log 2>&1 &
# sleep 20 && curl -s http://localhost:8080/ping
# agentcore dev --stream "Extract a tool call as JSON from: ..." </dev/null
# for p in $(pgrep -f 'agentcore dev'); do kill $p; done
```

## 3. 배포 (AWS Runtime endpoint)
로컬 검증이 끝나면 클라우드에 배포합니다. 리전/GA 여부/CLI 스키마를 다시 확인한 뒤 터미널에서 실행하세요.

In [ ]:
# 터미널에서 (Node ≥ 20: 위 0단계 스크립트로 설치, create_agent.sh로 생성/로컬검증 후):
#   export SLM_ENDPOINT_NAME=<endpoint> AWS_REGION=<리전> BEDROCK_CLAUDE_MODEL_ID=global.anthropic.claude-sonnet-5
#   cd agentcore/gemmaextraction
#   agentcore deploy                       # ARM64 → ECR → Runtime endpoint (CDK)
#   agentcore invoke --prompt '...'        # 배포된 endpoint 호출
print('배포는 위 주석 명령을 터미널에서: 리전/GA/CLI 스키마 재확인 후 실행하세요')

## 정리
AgentCore Runtime과 ECR 이미지, 그리고 여기에 연결된 SageMaker endpoint는 모두 과금 대상 리소스입니다. 실습을 마치면 99_cleanup으로 endpoint를 정리하고, AgentCore Runtime도 별도로 삭제하세요.